# Multi-Document Assistant using Hybrid Retrieval

This notebook demonstrates how to build a multi-document assistant using LangChain, Google Gemini, ChromaDB, and BM25 for hybrid retrieval. The assistant will answer questions based on a collection of PDF documents.

## 1. Setup and Installation

First, we install all the necessary Python libraries. This includes LangChain components, ChromaDB for vector storage, PyPDF for PDF loading, Rank BM25 for lexical search, and LangChain Google GenAI for interacting with Google's generative models and embeddings.

In [1]:
!pip install -q langchain langchain-community langchain-core chromadb pypdf rank_bm25 langchain-google-genai langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = [
    "/content/Legal-Services-Agreement.pdf",
    "/content/Contract_document.pdf"
]

documents = []

for path in pdf_paths:
    try:
        loader = PyPDFLoader(path)
        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = path.split("/")[-1]

        documents.extend(docs)
    except Exception as e:
        print(f"Error loading {path}: {e}")

print(f"Loaded {len(documents)} pages")

Loaded 38 pages


## 2. Document Loading and Preprocessing

We load the PDF documents using `PyPDFLoader`. Each page of the PDF is treated as a separate document, and we add the original filename as metadata.

### 2.1 Text Splitting

To manage context and improve retrieval accuracy, we split the loaded documents into smaller, overlapping chunks using `RecursiveCharacterTextSplitter`. This helps in breaking down large documents into manageable pieces while retaining context across splits.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

Created 90 chunks


## 3. Vector Database and Hybrid Retrieval Setup

This section sets up the vector database using ChromaDB and defines the hybrid retrieval mechanism, combining vector similarity search with BM25 lexical search.

### 3.1 `create_vector_db` Function

This function handles the creation or loading of the Chroma vector database. It includes logic for batching document additions to the database, which can be useful for large datasets. It checks if an existing database is present to avoid recreating it.

In [4]:
import os
import time
from langchain_chroma import Chroma

def create_vector_db(chunks, embeddings, persist_dir="chroma_db"):
    if os.path.exists(persist_dir):
        print("Loading existing DB...")
        return Chroma(
            persist_directory=persist_dir,
            embedding_function=embeddings
        )

    print("Creating new DB (batched)...")

    texts = [doc.page_content for doc in chunks]
    metas = [doc.metadata for doc in chunks]

    vector_db = None
    batch_size = 10

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_meta = metas[i:i+batch_size]

        print(f"Batch {i//batch_size + 1}")

        if vector_db is None:
            vector_db = Chroma.from_texts(
                texts=batch_texts,
                embedding=embeddings,
                metadatas=batch_meta,
                persist_directory=persist_dir
            )
        else:
            vector_db.add_texts(
                texts=batch_texts,
                metadatas=batch_meta
            )

        time.sleep(1)

    vector_db.persist()
    print("DB saved!")

    return vector_db

### 3.2 Initialize Embeddings and ChromaDB

We initialize the Google Generative AI embeddings model (`gemini-embedding-001`) and then use it to create or load our Chroma vector database from the pre-processed document chunks. The API key is securely fetched from Colab's secrets manager.

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# Fetch the API key from Colab's secrets manager
api_key = userdata.get('GEMINI_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

vector_db = create_vector_db(
    chunks,
    embeddings
)

Loading existing DB...


### 3.3 Initialize BM25 Lexical Search

BM25 (Best Match 25) is a ranking function used in information retrieval that estimates the relevance of documents to a given search query. Here, we tokenize our document chunks to prepare them for BM25 indexing.

In [7]:
from rank_bm25 import BM25Okapi

chunk_texts = [doc.page_content for doc in chunks]
tokenized_corpus = [text.split() for text in chunk_texts]

bm25 = BM25Okapi(tokenized_corpus)

### 3.4 Hybrid Retrieval Function

The `hybrid_retrieve` function combines both vector similarity search (using the ChromaDB) and BM25 lexical search. It retrieves top documents from both methods and then combines them, removing duplicates, to provide a more comprehensive set of relevant documents. This leverages the strengths of both semantic and keyword-based search.

In [8]:
def hybrid_retrieve(query, k=3):
    # Vector search
    vector_results = vector_db.similarity_search(query, k=k)

    # BM25 search
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True
    )[:k]

    bm25_results = [chunks[i] for i in top_indices]

    # Combine + remove duplicates
    combined = vector_results + bm25_results

    seen = set()
    final_docs = []

    for doc in combined:
        text = doc.page_content
        if text not in seen:
            seen.add(text)
            final_docs.append(doc)

    return final_docs[:k], vector_results, bm25_results

## 4. Language Model and Assistant Setup

This section configures the large language model (LLM) and defines the prompt structure for our legal assistant.

### 4.1 Initialize LLM and Prompt Template

We use `ChatGoogleGenerativeAI` with the `gemini-2.5-flash` model as our LLM. A `ChatPromptTemplate` is defined to instruct the LLM to act as a legal assistant, answering strictly based on the provided context from the documents.

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=api_key
)

prompt = ChatPromptTemplate.from_template("""
You are a legal assistant.

Answer strictly based on the contract.

- Identify relevant clauses
- Highlight obligations or risks if present
- Be precise and professional
- If not found, say: Not found in document

Context:
{context}

Question:
{question}
""")

In [10]:
import time

def safe_llm_call(prompt_template, context, question):
    time.sleep(3)  # prevents burst quota exhaustion
    return llm.invoke(prompt_template.format(
        context=context,
        question=question
    ))

### 4.2 `ask_legal_assistant` Function

This function orchestrates the entire query process. It takes a user query, uses the `hybrid_retrieve` function to get relevant document chunks, formats them into a context, and then invokes the LLM with the context and the user's question. It returns the LLM's answer along with the retrieved documents for source attribution, and the separate vector and BM25 results.

In [11]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

def ask_legal_assistant(query):
    # Get intent and corresponding pipeline settings
    classified_intent = classify_intent(query)
    pipeline_settings = intent_pipelines.get(classified_intent, intent_pipelines["General Inquiry"])

    # Retrieve documents using the k_value from the pipeline settings
    docs, vector_res, bm25_res = hybrid_retrieve(query, k=pipeline_settings["k_value"])
    context = format_docs(docs)

    # Use the specific prompt template for the intent
    prompt_template = pipeline_settings["prompt_template"]

    # Invoke LLM with safe call and specific prompt
    response = safe_llm_call(prompt_template, context, query)

    return response.content, docs, vector_res, bm25_res

## 5. Ask a Question

Now, let's ask a question to our legal assistant and see its response, including the sources it used.

# Adding more features:
## (Phase 1) Implementing Advanced Intent Routing

### Right now your system is:

one query → hybrid retrieval → one prompt

### We will upgrade it to:

query → intent → specialized pipeline

## Define Intents and Examples


In [12]:
intents_and_examples = {
    "Termination Clause Query": [
        "What are the conditions for contract termination?",
        "How can this agreement be terminated?",
        "Tell me about the termination clause.",
        "What happens if either party wants to end the contract?",
        "Summarize the termination procedures."
    ],
    "Definition Request": [
        "Define 'Institute'.",
        "What does 'service provider' mean in this document?",
        "What is the definition of 'confidential information'?",
        "Explain the term 'effective date'.",
        "Can you clarify 'Force Majeure'?"
    ],
    "Obligation Identification": [
        "What are the obligations of the service provider?",
        "What responsibilities does the Institute have?",
        "List the duties of each party.",
        "What is required from the client?",
        "What must the contractor do under this agreement?"
    ],
    "Payment Terms Inquiry": [
        "How are payments handled?",
        "What are the payment terms?",
        "When is payment due?",
        "Is there any information about late payment penalties?",
        "What is the payment schedule?"
    ],
    "Dispute Resolution": [
        "How are disputes resolved?",
        "What is the process for settling disagreements?",
        "Where does arbitration take place?",
        "What governing law applies to this contract?",
        "What if there's a conflict between the parties?"
    ],
    "Intellectual Property Rights": [
        "Who owns the intellectual property created?",
        "What are the IP rights under this agreement?",
        "Is there a clause about copyrights or patents?",
        "What happens to work product ownership?",
        "Describe the intellectual property provisions."
    ]
}

for intent, queries in intents_and_examples.items():
    print(f"Intent: {intent}")
    for i, query in enumerate(queries):
        print(f"  {i+1}. {query}")
    print()


Intent: Termination Clause Query
  1. What are the conditions for contract termination?
  2. How can this agreement be terminated?
  3. Tell me about the termination clause.
  4. What happens if either party wants to end the contract?
  5. Summarize the termination procedures.

Intent: Definition Request
  1. Define 'Institute'.
  2. What does 'service provider' mean in this document?
  3. What is the definition of 'confidential information'?
  4. Explain the term 'effective date'.
  5. Can you clarify 'Force Majeure'?

Intent: Obligation Identification
  1. What are the obligations of the service provider?
  2. What responsibilities does the Institute have?
  3. List the duties of each party.
  4. What is required from the client?
  5. What must the contractor do under this agreement?

Intent: Payment Terms Inquiry
  1. How are payments handled?
  2. What are the payment terms?
  3. When is payment due?
  4. Is there any information about late payment penalties?
  5. What is the payme

In [13]:
from langchain_core.prompts import ChatPromptTemplate

intent_list = "\n- ".join(intents_and_examples.keys())

intent_prompt = ChatPromptTemplate.from_template("""
You are an intent classification system.
Given a user query, classify it into one of the following intents. Respond ONLY with the intent name, nothing else.

Available Intents:
- {intent_list}

Query: {query}

Intent:
""")

def classify_intent(query):
    global llm
    time.sleep(3) # Add sleep before LLM call for intent classification
    response = llm.invoke(intent_prompt.format(
        intent_list=intent_list,
        query=query
    ))
    classified_intent = response.content.strip()

    # Basic error handling: if the LLM returns an unexpected intent, default to 'General Inquiry'
    if classified_intent not in intents_and_examples:
        print(f"Warning: LLM returned unrecognized intent: {classified_intent}. Defaulting to 'General Inquiry'.")
        return "General Inquiry"
    return classified_intent

print("Intent classification prompt and function defined.")

Intent classification prompt and function defined.


In [14]:
from langchain_core.prompts import ChatPromptTemplate

# Define specialized prompts for each intent

# Existing general prompt (for 'General Inquiry' and as a fallback)
generic_legal_prompt = ChatPromptTemplate.from_template("""
You are a legal assistant.

Answer strictly based on the contract.

- Identify relevant clauses
- Highlight obligations or risks if present
- Be precise and professional
- If not found, say: Not found in document

Context:
{context}

Question:
{question}
""")

termination_prompt = ChatPromptTemplate.from_template("""
You are a legal assistant focused on contract termination clauses.

Based on the provided context, identify and summarize all conditions, procedures, and implications related to contract termination. Highlight any notice periods, reasons for termination, and financial consequences.
If not found, say: Not found in document

Context:
{context}

Question:
{question}
""")

definition_prompt = ChatPromptTemplate.from_template("""
You are a legal assistant. Your task is to provide a concise and clear definition of the term(s) requested, based strictly on the provided contract context.
If the term is not explicitly defined, state that it is 'Not found in document'.

Context:
{context}

Question:
{question}
""")

obligations_prompt = ChatPromptTemplate.from_template("""
You are a legal assistant. Based on the provided context, list all obligations and responsibilities for the parties mentioned in the question (e.g., 'service provider', 'Institute'). Clearly state which party has which obligation.
If not found, say: Not found in document

Context:
{context}

Question:
{question}
""")

payment_prompt = ChatPromptTemplate.from_template("""
You are a legal assistant focused on payment terms. Based on the provided context, describe the payment schedule, due dates, methods, and any penalties or conditions related to payments.
If not found, say: Not found in document

Context:
{context}

Question:
{question}
""")

dispute_resolution_prompt = ChatPromptTemplate.from_template("""
You are a legal assistant focused on dispute resolution. Based on the provided context, explain the process for resolving disputes, including any applicable governing law, arbitration, or mediation clauses.
If not found, say: Not found in document

Context:
{context}

Question:
{question}
""")

ip_rights_prompt = ChatPromptTemplate.from_template("""
You are a legal assistant focused on intellectual property rights. Based on the provided context, describe the provisions related to intellectual property ownership, licensing, usage, and protection.
If not found, say: Not found in document

Context:
{context}

Question:
{question}
""")

# Define intent pipelines with k_value and prompt_template
intent_pipelines = {
    "General Inquiry": {
        "k_value": 3,
        "prompt_template": generic_legal_prompt
    },
    "Termination Clause Query": {
        "k_value": 5, # More context for comprehensive termination details
        "prompt_template": termination_prompt
    },
    "Definition Request": {
        "k_value": 2, # Definitions are usually concise and localized
        "prompt_template": definition_prompt
    },
    "Obligation Identification": {
        "k_value": 5, # Obligations can be spread across various sections
        "prompt_template": obligations_prompt
    },
    "Payment Terms Inquiry": {
        "k_value": 4, # Payment terms might include several related clauses
        "prompt_template": payment_prompt
    },
    "Dispute Resolution": {
        "k_value": 4, # Dispute resolution can involve multiple steps and clauses
        "prompt_template": dispute_resolution_prompt
    },
    "Intellectual Property Rights": {
        "k_value": 4, # IP clauses can be detailed and cover different aspects
        "prompt_template": ip_rights_prompt
    }
}

print("Specialized pipelines (k_value and prompt_template) designed for each intent.")

Specialized pipelines (k_value and prompt_template) designed for each intent.


## Test updated ask_legal_assistant function with a variety of sample queries, each designed to trigger a different intent.


In [15]:
test_queries = {
    # "Termination Clause Query": "What are the conditions for contract termination?",
    "Definition Request": "Define 'Institute'.",
    # "Obligation Identification": "What are the responsibilities of the service provider?",
    # "Payment Terms Inquiry": "How are payments handled under this agreement?",
    # "Dispute Resolution": "How are disputes resolved in this contract?",
    # "Intellectual Property Rights": "Who owns the intellectual property created?",
    # "General Inquiry": "What is the general purpose of this legal document?"
}

for expected_intent, query_text in test_queries.items():
    print(f"\n------- Testing Query: {query_text} -------")
    print(f"Expected Intent: {expected_intent}")

    # Classify intent to see what k_value and prompt will be used
    classified_intent = classify_intent(query_text)
    print(f"Classified Intent: {classified_intent}")

    # Get the answer and sources using the dynamically configured pipeline
    answer, docs, vector_results, bm25_results = ask_legal_assistant(query_text)

    print("\nANSWER:")
    print(answer)

    print("\nSOURCES:")
    seen_sources = set()
    for doc in docs:
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "N/A")
        key = (source, page)
        if key not in seen_sources:
            print(f"{source} (Page {page})")
            seen_sources.add(key)



------- Testing Query: Define 'Institute'. -------
Expected Intent: Definition Request
Classified Intent: Definition Request

ANSWER:
The "Institute" shall mean the Indian Institute of Technology Kanpur (IITK) with its premises located at Kalyanpur, Kanpur 208016 and shall include its authorized representatives, successors and assignees.

SOURCES:
Contract_document.pdf (Page 3)
Contract_document.pdf (Page 23)
